# 03 · The tail, and how concentrated it is

**Question.** Does the average describe the passenger experience?

Two statistics answer it: the percentile at which the mean sits, and a Lorenz curve
over positive delay minutes.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import tail, viz

panel = clean.completed(io.load_flights())

tail.percentile_profile(panel)

## 3.1 Concentration

The curve is read by position, which is the most accurate encoding available. The
top-1% share is written on the figure because a reader would otherwise estimate it.

In [ ]:
concentration = tail.lorenz(panel, sample_rows=2_000_000)
print(f"Gini-style coefficient {concentration['gini']:.3f}")
print(f"worst 1% of flights   -> {concentration['top1_share']:.1%} of delay minutes")
print(f"worst 5% of flights   -> {concentration['top5_share']:.1%} of delay minutes")

viz.fig_lorenz(concentration)

## 3.2 Does the mean hide the tail *between* months?

The pilot found r = 0.998 across months: the mean and the severe share move
together at that level of aggregation. That is a null result, and it is reported
rather than dropped — the "mean hides the tail" claim holds for individual flights,
not between monthly averages.

In [ ]:
paradox = tail.mean_vs_tail(panel, by="month")
print(f"Pearson r  {paradox['pearson_r']:+.3f}")
print(f"Spearman r {paradox['spearman_r']:+.3f}")
print(f"verdict    {paradox['verdict']}")

paradox["table"]

In [ ]:
viz.fig_percentile_band(tail.monthly_profile(panel))

## Visual audit — Cairo's five questions

### Figure 04 · The average is not the typical flight

| Cairo question | Answer |
|---|---|
| **Guidelines** | A percentile band around the median rather than a mean line: the reader sees the distribution's shape, not a single number standing in for it. Shaded band plus one median line, no dual axis. |
| **Truthful** | The median sits below zero in every month while the mean sits above it. Keeping both visible is the point: a chart that showed only the mean would suggest a uniformly late system, and one that showed only the median would suggest no problem at all. Both facts are in `tail.json`. |
| **Functional** | The reader can locate a month, read the median and the 10th–90th range, and understand why "average delay" and "typical flight" disagree. Axis labels carry units. |
| **Beautiful** | One hue, two tints, no gridlines behind the band, and the zero line emphasised because zero is the meaningful reference, not the axis floor. |
| **Insightful** | 34.4% of flights arrive late while the median arrives six minutes early, and 18.6% arrive 15+ minutes late. The figure makes the asymmetry visible before the text explains it. |

### Figure 05 · Lorenz curve of positive delay minutes

| Cairo question | Answer |
|---|---|
| **Guidelines** | Cumulative share of minutes against cumulative share of flights, both 0–100%, with the equality diagonal drawn as a reference. A reference line is what turns a curve into a comparison. |
| **Truthful** | The curve is built on positive delay minutes only, stated in the note; early flights cannot contribute negative area. The Gini of 0.8757 is printed with the curve rather than left for the reader to infer from the gap. |
| **Functional** | A reader can read off the concentration statement directly: the worst 1% of flights carry 25.0% of positive minutes, the worst 5% carry 58.3%. |
| **Beautiful** | Two lines, one fill, one label — the minimum ink that still supports the comparison. The diagonal is dashed and grey so it never competes with the data. |
| **Insightful** | The curve is what makes "mean 13.3 minutes" and "median −6.0 minutes" reconcile: the burden sits in a small tail, so any policy aimed at the average misses the flights that carry the delay. |

### Figure 14 · One mean per month versus the same months with the distribution kept

| Cairo question | Answer |
|---|---|
| **Guidelines** | A deliberate pair. Panel A is the weaker encoding (one mean per month), panel B is the same data with the distribution retained. Putting them side by side makes the encoding choice the subject of the figure. |
| **Truthful** | Both panels use the same twelve month aggregates and the same vertical scale family; panel A is not a strawman built from a different sample. The note says the correlation between mean and tail is computed on twelve pooled aggregates, not on individual flights. |
| **Functional** | The reader is asked to notice what panel A hides — the median never crosses zero — and can verify it in panel B without leaving the page. |
| **Beautiful** | Identical typography and colour in both panels, so the difference the reader sees is the encoding, not the styling. |
| **Insightful** | It is a self-critique: the project's own earlier chart form is shown next to its replacement. That is the strongest available evidence that the visual audit changed a decision rather than decorating one. |